In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers keybert umap-learn hdbscan razdel pymorphy3 tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 50.5 MB/s eta 0:00:00


In [3]:
import json
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from razdel import sentenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import umap
import hdbscan
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [4]:
FULL_DATA_PATH = "/content/drive/MyDrive/term_paper_masha/bank_reviews_dataset.csv"
df = pd.read_csv(FULL_DATA_PATH, sep=";", encoding="utf-8-sig")

print("Размер датасета:", df.shape)
print("Столбцы:")
print(df.columns.tolist())

Размер датасета: (6398, 8)
Столбцы:
['id', 'link', 'date', 'title', 'text', 'rating', 'city', 'source']


In [5]:
expected_cols = ["id", "link", "date", "title", "text", "rating", "city", "source"]
df = df[expected_cols].copy()
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["review_text"] = (df["title"].fillna("").astype(str)+  ". " + df["text"].fillna("").astype(str)
)

In [6]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["review_text_clean"] = df["review_text"].apply(clean_text)
df = df[df["review_text_clean"].str.len() > 50].copy()
df = df.drop_duplicates(subset=["review_text_clean"]).reset_index(drop=True)
df["text_len"] = df["review_text_clean"].str.len()
print(df.shape)
df[["source", "rating", "text_len", "review_text_clean"]].head()

(6398, 11)


,source,rating,text_len,review_text_clean
0,banki,1.0,3095,"Блокировка карты вместе с деньгами, бросание т..."
1,banki,2.0,578,Самый ужасный банк в обслуживании. Самый ужасн...
2,banki,1.0,1099,"Граждане, бегите из этого банка, если хотите, ..."
3,banki,1.0,2606,Как Газпромбанк получает млрд на деньгах гражд...
4,banki,1.0,575,"Блокировка карты. Заказал, дебетовую карту Газ..."


RuBERT

In [7]:
model = SentenceTransformer("cointegrated/rubert-tiny2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
def split_into_sentences(text):
    sentences = [s.text.strip() for s in sentenize(text)]
    sentences = [s for s in sentences if len(s) > 20]
    return sentences


def make_chunks(sentences, max_chars=900):
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) <= max_chars:
            current += " " + sent
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sent
    if current.strip():
        chunks.append(current.strip())

    return chunks

In [9]:
def reduce_long_review(text, model, max_chunks=4, max_chars=900):
    text = clean_text(text)
    if len(text) <= 2500:
        return text

    sentences = split_into_sentences(text)
    if len(sentences) <= 5:
        return text[:2500]
    chunks = make_chunks(sentences, max_chars=max_chars)
    if len(chunks) <= max_chunks:
        return " ".join(chunks)

    query = " ".join(sentences[:3])
    chunk_emb = model.encode(chunks, normalize_embeddings=True)
    query_emb = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_emb, chunk_emb)[0]
    top_idx = np.argsort(scores)[-max_chunks:]
    top_idx = sorted(top_idx)
    selected_chunks = [chunks[i] for i in top_idx]

    return " ".join(selected_chunks)

In [10]:
tqdm.pandas()

df["review_short"] = df["review_text_clean"].progress_apply(lambda x: reduce_long_review(x, model=model))
df["short_len"] = df["review_short"].str.len()
df[["text_len", "short_len", "review_short"]].head()

  0%|          | 0/6398 [00:00<?, ?it/s]

,text_len,short_len,review_short
0,3095,3009,"Блокировка карты вместе с деньгами, бросание т..."
1,578,578,Самый ужасный банк в обслуживании. Самый ужасн...
2,1099,1099,"Граждане, бегите из этого банка, если хотите, ..."
3,2606,2543,Как Газпромбанк получает млрд на деньгах гражд...
4,575,575,"Блокировка карты. Заказал, дебетовую карту Газ..."


In [11]:
texts = df["review_short"].tolist()

embeddings = model.encode(texts,batch_size=64,show_progress_bar=True,normalize_embeddings=True)
embeddings.shape

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

(6398, 312)

In [12]:
reducer = umap.UMAP(n_neighbors=15,n_components=5,min_dist=0.0,metric="cosine",random_state=42)

embeddings_umap = reducer.fit_transform(embeddings)
embeddings_umap.shape

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(6398, 5)

In [13]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=10,min_samples=3,metric="euclidean",prediction_data=True)
df["cluster_topic_id"] = clusterer.fit_predict(embeddings_umap)
df["cluster_topic_id"].value_counts().sort_index()

,count
cluster_topic_id,
-1,517
0,5153
1,21
2,18
3,13
4,19
5,16
6,40
7,21


In [14]:
noise_share = (df["cluster_topic_id"] == -1).mean()
print("Noise share:", noise_share)
if noise_share > 0.5:
    print("Too much noise. Using KMeans instead.")
    n_clusters = 12
    kmeans = KMeans(n_clusters=n_clusters,random_state=42,n_init="auto")
    df["cluster_topic_id"] = kmeans.fit_predict(embeddings_umap)

df["cluster_topic_id"].value_counts().sort_index()

Noise share: 0.08080650203188497


,count
cluster_topic_id,
-1,517
0,5153
1,21
2,18
3,13
4,19
5,16
6,40
7,21


In [15]:
LABELS = [
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
]

In [16]:
services = {
    "CARDS": [
        "карта", "дебетовая карта", "кредитная карта", "кредитка",
        "пин", "пин-код", "кэшбек", "кэшбэк", "кешбек", "кешбэк",
        "перевыпуск", "доставка карты", "блокировка карты"
    ],

    "DIGITAL": [
        "приложение", "мобильное приложение", "интернет-банк",
        "личный кабинет", "онлайн банк", "авторизация", "вход",
        "смс код", "код подтверждения", "ошибка приложения"
    ],

    "SUPPORT": [
        "поддержка", "чат", "оператор", "горячая линия",
        "контактный центр", "служба поддержки", "звонок", "обратная связь"
    ],

    "PAYMENTS": [
        "перевод", "платеж", "платёж", "оплата", "сбп",
        "транзакция", "операция", "зачисление", "списание", "вывод денег"
    ],

    "CREDITS": [
        "кредит", "кредитка", "ипотека", "автокредит",
        "займ", "платеж по кредиту", "график платежей",
        "досрочное погашение"
    ],

    "FEES": [
        "комиссия", "комиссию", "плата", "платно",
        "годовое обслуживание", "стоимость обслуживания",
        "списали деньги", "удержали деньги"
    ],

    "OFFICE": [
        "офис", "отделение", "касса", "кассир",
        "сотрудник офиса", "очередь", "талон", "обслуживание в офисе"
    ],

    "ATM": [
        "банкомат", "снятие наличных", "снять наличные",
        "внести наличные", "пополнение через банкомат",
        "банкомат съел", "банкомат не выдал"
    ],

    "FRAUD": [
        "мошенник", "мошенничество", "антифрод", "115-фз",
        "115 фз", "финмониторинг", "заблокировали счет",
        "заблокировали карту", "служба безопасности"
    ],

    "INSURANCE": [
        "страховка", "страхование", "осаго", "каско",
        "страхование жизни", "страхование карты", "ипотечная страховка"
    ],

    "INVESTMENTS": [
        "инвестиции", "инвестиция", "брокер", "брокерский счет",
        "иис", "пиф", "ценные бумаги", "акции", "облигации", "накопительный счёт", "вклад"
    ],

    "PREMIUM": [
        "премиум", "premium", "private banking",
        "премиальное обслуживание", "персональный менеджер",
        "привилегия", "премиальный клиент"
    ]
}

In [17]:
positive_words = [
    "хорошо", "отлично", "прекрасно", "удобно", "быстро",
    "понравилось", "доволен", "довольна", "спасибо",
    "помогли", "решили", "оперативно", "качественно",
    "вежливо", "грамотно", "выгодно", "рекомендую",
    "лучший", "приятно", "без проблем"
]

negative_words = [
    "плохо", "ужасно", "отвратительно", "не работает",
    "не помогли", "не решили", "проблема", "ошибка",
    "долго", "хамство", "обман", "навязали",
    "списали", "заблокировали", "не вернули",
    "не пришли", "не начислили", "невозможно",
    "отказ", "жалоба", "претензия", "разочарован",
    "разочарована"
]

In [18]:
def normalize_for_rules(text):
    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^а-яa-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


services_norm = {service: [normalize_for_rules(keyword) for keyword in keywords]for service, keywords in services.items()}
positive_words_norm = [normalize_for_rules(w) for w in positive_words]
negative_words_norm = [normalize_for_rules(w) for w in negative_words]

In [19]:
def find_services_multilabel(text, services_dict=services_norm):
    text_norm = normalize_for_rules(text)
    found = {}
    for service, keywords in services_dict.items():
        score = 0
        matched_keywords = []
        for keyword in keywords:
            if keyword in text_norm:
                score += len(keyword.split())
                matched_keywords.append(keyword)
        if score > 0:
            found[service] = {"score": score, "matched_keywords": matched_keywords}

    return found

In [20]:
def define_sentiment(text, rating=None):
    text_norm = normalize_for_rules(text)
    pos_score = sum(1 for word in positive_words_norm if word in text_norm)
    neg_score = sum(1 for word in negative_words_norm if word in text_norm)
    if pos_score > neg_score:
        return "POS"
    if neg_score > pos_score:
        return "NEG"
    if pd.notna(rating):
        if rating >= 4:
            return "POS"
        if rating <= 3:
            return "NEG"

    return "NEG"

In [21]:
ALL_LABELS = [
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
]

In [22]:
def classify_review_multilabel(row):
    text = row["review_short"]
    rating = row["rating"]
    sentences = split_into_sentences(text)
    if len(sentences) == 0:
        sentences = [text]
    labels = []
    matched_keywords_all = []
    for sent in sentences:
        found_services = find_services_multilabel(sent)
        if not found_services:
            continue

        sentiment = define_sentiment(sent, rating)
        for service, info in found_services.items():
            label = f"{service}_{sentiment}"
            if label in ALL_LABELS and label not in labels:
                labels.append(label)
                matched_keywords_all.extend(info["matched_keywords"])
    if not labels:
        sentiment = define_sentiment(text, rating)
        labels = [f"OTHER_{sentiment}"]

    return pd.Series({"labels": labels, "labels_str": ", ".join(labels), "matched_keywords": ", ".join(sorted(set(matched_keywords_all))[:30])})

In [23]:
df[["labels", "labels_str", "matched_keywords"]] = df.progress_apply( classify_review_multilabel, axis=1)
df[["rating", "cluster_topic_id", "labels_str", "matched_keywords", "review_short"]].head()

  0%|          | 0/6398 [00:00<?, ?it/s]

,rating,cluster_topic_id,labels_str,matched_keywords,review_short
0,1.0,0,"CARDS_NEG, ATM_NEG, FRAUD_NEG, SUPPORT_NEG, FR...","банкомат, блокировка карты, заблокировали карт...","Блокировка карты вместе с деньгами, бросание т..."
1,2.0,0,"CARDS_NEG, OFFICE_NEG","кэшбэк, офис",Самый ужасный банк в обслуживании. Самый ужасн...
2,1.0,0,INVESTMENTS_NEG,вклад,"Граждане, бегите из этого банка, если хотите, ..."
3,1.0,0,"PAYMENTS_NEG, FEES_NEG, CARDS_POS, FRAUD_NEG, ...","карта, оплата, плата, платеж, финмониторинг, чат",Как Газпромбанк получает млрд на деньгах гражд...
4,1.0,0,OFFICE_NEG,офис,"Блокировка карты. Заказал, дебетовую карту Газ..."


In [24]:
df_labels = df.explode("labels").reset_index(drop=True)
df_labels = df_labels.rename(columns={"labels": "label"})
df_labels[["id", "source", "date", "rating","cluster_topic_id", "label", "review_short", "matched_keywords"]].head(10)

,id,source,date,rating,cluster_topic_id,label,review_short,matched_keywords
0,12899382,banki,2026-02-08,1.0,0,CARDS_NEG,"Блокировка карты вместе с деньгами, бросание т...","банкомат, блокировка карты, заблокировали карт..."
1,12899382,banki,2026-02-08,1.0,0,ATM_NEG,"Блокировка карты вместе с деньгами, бросание т...","банкомат, блокировка карты, заблокировали карт..."
2,12899382,banki,2026-02-08,1.0,0,FRAUD_NEG,"Блокировка карты вместе с деньгами, бросание т...","банкомат, блокировка карты, заблокировали карт..."
3,12899382,banki,2026-02-08,1.0,0,SUPPORT_NEG,"Блокировка карты вместе с деньгами, бросание т...","банкомат, блокировка карты, заблокировали карт..."
4,12899382,banki,2026-02-08,1.0,0,FRAUD_POS,"Блокировка карты вместе с деньгами, бросание т...","банкомат, блокировка карты, заблокировали карт..."
5,12898481,banki,2026-02-08,2.0,0,CARDS_NEG,Самый ужасный банк в обслуживании. Самый ужасн...,"кэшбэк, офис"
6,12898481,banki,2026-02-08,2.0,0,OFFICE_NEG,Самый ужасный банк в обслуживании. Самый ужасн...,"кэшбэк, офис"
7,12898275,banki,2026-02-08,1.0,0,INVESTMENTS_NEG,"Граждане, бегите из этого банка, если хотите, ...",вклад
8,12898259,banki,2026-02-08,1.0,0,PAYMENTS_NEG,Как Газпромбанк получает млрд на деньгах гражд...,"карта, оплата, плата, платеж, финмониторинг, чат"
9,12898259,banki,2026-02-08,1.0,0,FEES_NEG,Как Газпромбанк получает млрд на деньгах гражд...,"карта, оплата, плата, платеж, финмониторинг, чат"


In [25]:
empty_label_reviews = df[df["labels"].isna() |(df["labels"].apply(lambda x: len(x) == 0 if isinstance(x, list) else True))].copy()
print("Количество отзывов с пустым лейблом:", empty_label_reviews.shape[0])

Количество отзывов с пустым лейблом: 0


In [26]:
labels_summary = (df_labels.groupby("label", as_index=False).agg(reviews_count=("id", "count"), avg_rating=("rating", "mean"),
        examples=("review_short", lambda x: " ||| ".join(x.head(3).astype(str).str[:300]))).sort_values("reviews_count", ascending=False).reset_index(drop=True))
labels_summary

,label,reviews_count,avg_rating,examples
0,SUPPORT_NEG,2950,1.193625,"Блокировка карты вместе с деньгами, бросание т..."
1,OFFICE_NEG,1756,1.169704,Самый ужасный банк в обслуживании. Самый ужасн...
2,CARDS_NEG,1638,1.205739,"Блокировка карты вместе с деньгами, бросание т..."
3,INVESTMENTS_NEG,1619,1.218036,"Граждане, бегите из этого банка, если хотите, ..."
4,PAYMENTS_NEG,1557,1.149100,Как Газпромбанк получает млрд на деньгах гражд...
5,FEES_NEG,1019,1.221786,Как Газпромбанк получает млрд на деньгах гражд...
6,DIGITAL_NEG,964,1.226141,Перевыпуск карты. Являюсь клиентом Газпромбанк...
7,CREDITS_NEG,937,1.163287,Премиальное обслуживание. Много лет являюсь пр...
8,CARDS_POS,721,4.552011,Как Газпромбанк получает млрд на деньгах гражд...
9,SUPPORT_POS,693,4.151515,Выбираю услуги по программе лояльности Газпром...


In [27]:
final_df = df[["id", "link", "date", "title", "text", "rating", "city", "source", "labels_str"]].copy()
final_df = final_df.rename(columns={"labels_str": "found_labels"})
final_df.head()

,id,link,date,title,text,rating,city,source,found_labels
0,12899382,https://www.banki.ru/services/responses/bank/r...,2026-02-08,"Блокировка карты вместе с деньгами, бросание т...",Сегодня при снятии (попытке снять свои деньги)...,1.0,Рязань,banki,"CARDS_NEG, ATM_NEG, FRAUD_NEG, SUPPORT_NEG, FR..."
1,12898481,https://www.banki.ru/services/responses/bank/r...,2026-02-08,Самый ужасный банк в обслуживании,Самый ужасный/ не гибкий банк Росии и крупнейш...,2.0,Москва,banki,"CARDS_NEG, OFFICE_NEG"
2,12898275,https://www.banki.ru/services/responses/bank/r...,2026-02-08,"Граждане, бегите из этого банка, если хотите, ...","В Газпромбанке решила открыть вклад , но так к...",1.0,Новочеркасск,banki,INVESTMENTS_NEG
3,12898259,https://www.banki.ru/services/responses/bank/r...,2026-02-08,Как Газпромбанк получает млрд на деньгах гражд...,"09.01.26 ГПБ заблокировал карты, я сразу предо...",1.0,Новочеркасск,banki,"PAYMENTS_NEG, FEES_NEG, CARDS_POS, FRAUD_NEG, ..."
4,12897965,https://www.banki.ru/services/responses/bank/r...,2026-02-07,Блокировка карты,"Заказал, дебетовую карту Газпромбанк а на сайт...",1.0,Барнаул,banki,OFFICE_NEG


In [28]:
output_path = "/content/drive/MyDrive/term_paper_masha/rubert_embeddings_dictionary_predictions.csv"
final_df.to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
print("Saved:", output_path)

Saved: /content/drive/MyDrive/term_paper_masha/rubert_embeddings_dictionary_predictions.csv


#Results


In [29]:
bert_path = "/content/drive/MyDrive/term_paper_masha/rubert_embeddings_dictionary_predictions.csv"
teacher_path = "/content/drive/MyDrive/term_paper_masha/predictions_clean_full_aspects.csv"
manual_path = "/content/drive/MyDrive/term_paper_masha/labeled_reviews_with_additional_with_labels_list.csv"

bert_df = pd.read_csv(bert_path, sep=";", encoding="utf-8-sig")
teacher_df = pd.read_csv(teacher_path, sep=";", encoding="utf-8-sig")
manual_df = pd.read_csv(manual_path, sep=";", encoding="utf-8-sig")

print("bert_df:", bert_df.shape)
print("teacher_df:", teacher_df.shape)
print("manual_df:", manual_df.shape)
print("bert columns:", bert_df.columns.tolist())
print("teacher columns:", teacher_df.columns.tolist())
print("manual columns:", manual_df.columns.tolist())

bert_df: (6398, 9)
teacher_df: (6398, 5)
manual_df: (311, 15)
bert columns: ['id', 'link', 'date', 'title', 'text', 'rating', 'city', 'source', 'found_labels']
teacher columns: ['id', 'title', 'text', 'prediction', 'labels_list']
manual columns: ['annotation_id', 'annotator', 'city', 'created_at', 'date', 'id', 'label', 'lead_time', 'link', 'rating', 'source', 'text', 'title', 'updated_at', 'labels_list']


In [32]:
def parse_labels(x):
    if pd.isna(x):
        return []
    x = str(x)
    labels = re.findall(r"[A-Z]+_(?:POS|NEG)", x)
    labels = [label for label in labels if label in ALL_LABELS]

    return sorted(set(labels))

In [33]:
bert_df["pred_labels"] = bert_df["found_labels"].apply(parse_labels)
teacher_df["true_labels"] = teacher_df["labels_list"].apply(parse_labels)
manual_df["true_labels"] = manual_df["labels_list"].apply(parse_labels)

In [34]:
eval_df = teacher_df[["id", "true_labels"]].merge(bert_df[["id", "pred_labels"]], on="id", how="inner")
print("Количество отзывов для сравнения:", eval_df.shape[0])

eval_df.head()

Количество отзывов для сравнения: 6398


,id,true_labels,pred_labels
0,12899382,"[CARDS_NEG, FEES_NEG, FRAUD_NEG, SUPPORT_NEG]","[ATM_NEG, CARDS_NEG, FRAUD_NEG, FRAUD_POS, SUP..."
1,12898481,"[CARDS_NEG, DIGITAL_NEG, FEES_NEG, OTHER_NEG, ...","[CARDS_NEG, OFFICE_NEG]"
2,12898275,"[FEES_NEG, OTHER_NEG, SUPPORT_NEG]",[INVESTMENTS_NEG]
3,12898259,"[CARDS_NEG, FEES_NEG, FRAUD_NEG, OFFICE_NEG, S...","[CARDS_POS, FEES_NEG, FRAUD_NEG, PAYMENTS_NEG,..."
4,12897965,"[CARDS_NEG, OFFICE_NEG, SUPPORT_NEG]",[OFFICE_NEG]


In [35]:
def soft_accuracy_no_extra_penalty(true_labels, pred_labels):
    true_set = set(true_labels)
    pred_set = set(pred_labels)

    if len(true_set) == 0:
        return np.nan

    return len(true_set.intersection(pred_set)) / len(true_set)

In [36]:
def calculate_one_dataset_metrics(base_df, pred_df):
    eval_df = base_df[["id", "true_labels"]].merge(
        pred_df[["id", "pred_labels"]],
        on="id",
        how="inner"
    )

    eval_df["soft_accuracy"] = eval_df.apply(
        lambda row: soft_accuracy_no_extra_penalty(row["true_labels"], row["pred_labels"]),
        axis=1
    )

    mlb = MultiLabelBinarizer(classes=ALL_LABELS)

    y_true = mlb.fit_transform(eval_df["true_labels"])
    y_pred = mlb.transform(eval_df["pred_labels"])

    metrics = {
        "Precision": precision_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "F1": f1_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "Exact Match Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Soft Accuracy": eval_df["soft_accuracy"].mean() * 100
    }

    return metrics, eval_df

In [39]:
teacher_metrics, teacher_eval_df = calculate_one_dataset_metrics(base_df=teacher_df, pred_df=bert_df)
manual_metrics, manual_eval_df = calculate_one_dataset_metrics(base_df=manual_df, pred_df=bert_df)

In [40]:
metrics_table = pd.DataFrame({"Qwen-generated labels": teacher_metrics, "Manual labels": manual_metrics}).round(2)

metrics_table

,Qwen-generated labels,Manual labels
Precision,50.13,42.36
Recall,42.51,51.36
F1,46.01,46.43
Exact Match Accuracy,3.16,7.07
Soft Accuracy,43.96,52.02
